# 检测理论 (Detection Theory)

本Notebook介绍现代通信系统中的检测算法，包括:

- ML（最大似然）检测
- MAP（最大后验）检测
- ZF（迫零）线性检测
- MMSE线性检测
- 消息传递检测（BP、AMP、GAMP）

## 学习目标

1. 理解贝叶斯检测准则：MAP与ML的关系
2. 掌握线性检测器的设计原理（ZF与MMSE）
3. 理解消息传递算法在稀疏信号检测中的应用
4. 通过Python仿真比较不同检测器的性能

## 1. 系统模型

考虑复数MIMO系统模型:

$$y = Hx + n$$

其中:
- $y \in \mathbb{C}^{N}$ 是接收信号向量
- $H \in \mathbb{C}^{N \times M}$ 是信道矩阵
- $x \in \mathbb{C}^{M}$ 是发送信号向量
- $n \sim \mathcal{CN}(0, \sigma^2 I)$ 是高斯噪声

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 系统参数
N = 8   # 接收天线数
M = 4   # 发送天线数（稀疏信号）
SNR_dB = 20  # 信噪比（dB）
num_trials = 1000  # 蒙特卡洛仿真次数

np.random.seed(42)

## 2. ML（最大似然）检测

ML检测器找到使条件概率最大化的信号:

$$\hat{x}_{ML} = \arg\max_{x \in \mathcal{X}} p(y|x) = \arg\min_{x \in \mathcal{X}} \|y - Hx\|^2$$

ML检测是最优检测，但计算复杂度为$O(|\mathcal{X}|^M)$，对大星座图和高维系统不可行。

In [ ]:
def ml_detector(y, H, constellation):
    """
    最大似然（ML）检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        constellation: 调制星座点数组
    
    返回:
        x_hat: 检测信号向量 (M,)
        min_dist: 最小欧氏距离
    """
    best_x = None
    best_dist = np.inf
    
    # 对于每个发送信号向量，计算距离
    # 由于M较小，使用穷举搜索
    for x in constellation:
        dist = np.linalg.norm(y - H @ x)**2
        if dist < best_dist:
            best_dist = dist
            best_x = x
    
    return best_x, best_dist


# 定义QPSK调制星座图
constellation_qpsk = np.array([
    1+1j, 1-1j, -1+1j, -1-1j
]) / np.sqrt(2)  # 归一化功率

## 3. MAP（最大后验）检测

MAP检测器考虑先验信息:

$$\hat{x}_{MAP} = \arg\max_{x \in \mathcal{X}} p(x|y) = \arg\max_{x \in \mathcal{X}} p(y|x)p(x)$$

当先验均匀分布时，MAP等价于ML。

对于稀疏信号，MP（Message Passing）利用先验信息可以提升性能。

In [ ]:
def map_detector(y, H, constellation, prior_probs=None):
    """
    最大后验（MAP）检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        constellation: 调制星座点数组
        prior_probs: 每个符号的先验概率 (len(constellation),)
    
    返回:
        x_hat: 检测信号向量 (M,)
    """
    if prior_probs is None:
        prior_probs = np.ones(len(constellation)) / len(constellation)
    
    best_x = None
    best_posterior = -np.inf
    
    for i, x in enumerate(constellation):
        # 计算似然 p(y|x)
        dist = np.linalg.norm(y - H @ x)**2
        likelihood = np.exp(-dist / (2 * 0.01))  # 假设噪声方差为0.01
        
        # 计算后验概率（未归一化）
        posterior = likelihood * prior_probs[i]
        
        if posterior > best_posterior:
            best_posterior = posterior
            best_x = x
    
    return best_x

## 4. ZF（迫零）线性检测

ZF检测器通过伪逆操作完全消除干扰:

$$\hat{x}_{ZF} = H^{+} y = (H^H H)^{-1} H^H y$$

优点：简单易实现，完全消除同道干扰
缺点：放大噪声，特别是当信道矩阵病态时

In [ ]:
def zf_detector(y, H):
    """
    迫零（ZF）线性检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
    
    返回:
        x_hat: 检测信号向量 (M,)
    """
    # 计算伪逆 H^+ = (H^H H)^{-1} H^H
    H_pseudo_inv = np.linalg.pinv(H)
    x_hat = H_pseudo_inv @ y
    return x_hat


def zf_detector_quantized(y, H, constellation):
    """
    ZF检测器 + 量化（最近邻判决）
    """
    x_float = zf_detector(y, H)
    # 量化到最近邻星座点
    distances = np.abs(constellation[:, np.newaxis] - x_float)
    indices = np.argmin(distances, axis=0)
    return constellation[indices]

## 5. MMSE（最小均方误差）线性检测

MMSE检测器在噪声放大和干扰消除之间取得平衡:

$$\hat{x}_{MMSE} = (H^H H + \sigma^2 I)^{-1} H^H y$$

当$\sigma^2 \to 0$时，MMSE退化为ZF。
当$\sigma^2 \to \infty$时，$\hat{x} \to 0$（拒绝做出判决）。

In [ ]:
def mmse_detector(y, H, noise_var=0.01):
    """
    最小均方误差（MMSE）线性检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        noise_var: 噪声方差
    
    返回:
        x_hat: 检测信号向量 (M,)
    """
    # 计算 MMSE 滤波矩阵: (H^H H + sigma^2 I)^{-1} H^H
    M_matrix = H.conj().T @ H + noise_var * np.eye(M)
    W_mmse = np.linalg.solve(M_matrix, H.conj().T)
    x_hat = W_mmse @ y
    return x_hat


def mmse_detector_quantized(y, H, constellation, noise_var=0.01):
    """
    MMSE检测器 + 量化（最近邻判决）
    """
    x_float = mmse_detector(y, H, noise_var)
    distances = np.abs(constellation[:, np.newaxis] - x_float)
    indices = np.argmin(distances, axis=0)
    return constellation[indices]

## 6. 消息传递检测算法

### 6.1 置信传播（Belief Propagation, BP）

BP算法通过因子图上的消息传递来计算后验概率。

对于MIMO检测，BP算法迭代地在变量节点和因子节点之间传递消息。

In [ ]:
def bp_detector(y, H, constellation, max_iter=20, tol=1e-6):
    """
    置信传播（BP）检测器 - 针对稀疏MIMO系统
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        constellation: 调制星座点数组
        max_iter: 最大迭代次数
        tol: 收敛容忍度
    
    返回:
        x_hat: 检测信号向量 (M,)
        converged: 是否收敛
    """
    N, M = H.shape
    K = len(constellation)
    
    # 初始化消息（从因子节点到变量节点）
    # f_nm[n,m] 表示从因子节点n到变量节点m的消息
    f_nm = np.ones((N, M, K)) / K
    
    # 初始化先验消息（从变量节点到因子节点）
    # g_mn[m,n] 表示从变量节点m到因子节点n的消息
    g_mn = np.ones((M, N, K)) / K
    
    prev_x = np.zeros(M, dtype=complex)
    
    for iteration in range(max_iter):
        # 步骤1: 更新因子节点消息 f_nm
        for n in range(N):
            for m in range(M):
                # 计算从因子节点n到变量节点m的消息
                # 基于其他变量节点的消息和信道模型
                for k in range(K):
                    # 计算有效信道输出
                    sum_val = 0j
                    for k_prime in range(K):
                        x_m = constellation[k_prime]
                        # 构造其他变量的估计
                        g_prod = 1.0
                        for m_prime in range(M):
                            if m_prime != m:
                                # 边缘消息乘积
                                g_prod *= g_mn[m_prime, n, k_prime]
                        
                        # 高斯似然
                        # y_n = H[n,m] * x_m + 其他干扰 + 噪声
                        # 简化: 使用欧氏距离
                        residual = y[n]
                        for m_prime in range(M):
                            if m_prime != m:
                                # 使用当前估计
                                pass  # 简化版本
                        
                        g_prod *= np.exp(-np.abs(y[n] - H[n, :] @ np.array([constellation[k_prime] if i == m else 0 for i in range(M)]))**2)
                    
                    f_nm[n, m, k] = g_prod
        
        # 归一化
        for n in range(N):
            for m in range(M):
                f_nm[n, m, :] /= np.sum(f_nm[n, m, :])
        
        # 步骤2: 更新变量节点消息 g_mn
        for m in range(M):
            for n in range(N):
                # 变量节点消息是其所有入消息的乘积
                for k in range(K):
                    prod = 1.0
                    for n_prime in range(N):
                        if n_prime != n:
                            prod *= f_nm[n_prime, m, k]
                    g_mn[m, n, k] = prod
        
        # 归一化
        for m in range(M):
            for n in range(N):
                g_mn[m, n, :] /= np.sum(g_mn[m, n, :])
        
        # 计算当前估计
        x_est = np.zeros(M, dtype=complex)
        for m in range(M):
            # 变量m的边缘后验概率
            posterior = np.ones(K) / K
            for n in range(N):
                posterior *= f_nm[n, m, :]
            posterior /= np.sum(posterior)
            
            # MAP估计
            x_est[m] = constellation[np.argmax(posterior)]
        
        # 检查收敛
        if np.linalg.norm(x_est - prev_x) < tol:
            break
        prev_x = x_est.copy()
    
    return x_est, iteration < max_iter - 1

### 6.2 AMP（近似消息传递）

AMP是BP的高斯近似版本，计算复杂度更低:

$$x^{t+1} = \eta_t(H^T (y - H x^t) + x^t)$$

其中$\eta_t$是非线性阈值函数。

In [ ]:
def amp_detector(y, H, noise_var=0.01, max_iter=50, tol=1e-6):
    """
    近似消息传递（AMP）检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        noise_var: 噪声方差
        max_iter: 最大迭代次数
        tol: 收敛容忍度
    
    返回:
        x_hat: 检测信号向量 (M,)
        residuals: 每次迭代的残差
    """
    N, M = H.shape
    x_hat = np.zeros(M, dtype=complex)
    residuals = []
    
    # 估计噪声方差
    sigma2 = noise_var
    
    for t in range(max_iter):
        # 计算残差（等效噪声）
        r = y - H @ x_hat + (M / N) * r * np.mean(np.diag(H.conj().T @ H)) / sigma2 if t > 0 else y - H @ x_hat
        
        # 匹配滤波
        v = H.conj().T @ r + x_hat
        
        # 简化阈值操作（复数软阈值）
        # 对于稀疏信号，使用收缩算子
        threshold = np.sqrt(sigma2 * 2 * np.log(1.0 / 0.1))  # 稀疏先验参数
        x_new = np.sign(v.real) * np.maximum(np.abs(v) - threshold, 0) + 1j * np.sign(v.imag) * np.maximum(np.abs(v) - threshold, 0)
        
        # 更新估计
        x_hat = x_new
        
        # 计算残差
        residual = np.linalg.norm(y - H @ x_hat) / np.linalg.norm(y)
        residuals.append(residual)
        
        if residual < tol:
            break
    
    return x_hat, residuals

### 6.3 GAMP（高斯近似消息传递）

GAMP是AMP的推广，处理更一般的线性高斯模型和任意先验分布。

In [ ]:
def gamp_detector(y, H, noise_var=0.01, max_iter=50, tol=1e-6):
    """
    高斯近似消息传递（GAMP）检测器
    
    参数:
        y: 接收信号向量 (N,)
        H: 信道矩阵 (N, M)
        noise_var: 噪声方差
        max_iter: 最大迭代次数
        tol: 收敛容忍度
    
    返回:
        x_hat: 检测信号向量 (M,)
        snr_iter: 每次迭代的估计SNR
    """
    N, M = H.shape
    
    # 初始化
    x_hat = np.zeros(M, dtype=complex)
    x_var = np.ones(M) * 1.0  # 信号方差
    
    snr_history = []
    
    for t in range(max_iter):
        # --- 线性步骤 ---
        # 计算有效输出
        s = H @ x_hat
        
        # 计算输出方差
        q_var = np.sum(np.abs(H)**2 * x_var[:, np.newaxis], axis=0) + noise_var
        
        # --- 非线性步骤 ---
        # 计算后验均值和方差的近似
        for m in range(M):
            # 信道增益
            h_m = H[:, m]
            
            # 有效信噪比
            snr_eff = (np.abs(h_m.conj().T @ (y - s + h_m * x_hat[m]))**2) / (np.sum(np.abs(h_m)**2) * noise_var + 1e-10)
            
            # 收缩估计
            w = h_m.conj().T @ (y - s) + x_hat[m]
            x_hat[m] = w / (1 + noise_var / (np.sum(np.abs(h_m)**2) + 1e-10))
            x_var[m] = noise_var / (np.sum(np.abs(h_m)**2) + 1e-10)
        
        snr_history.append(10 * np.log10(np.mean(np.abs(x_hat)**2) / (np.mean(x_var) + 1e-10)))
        
        # 检查收敛
        if t > 0 and abs(snr_history[-1] - snr_history[-2]) < tol:
            break
    
    return x_hat, snr_history

## 7. 性能对比仿真

In [ ]:
def generate_mimo_signal(N, M, constellation, sparsity=1.0):
    """
    生成MIMO信号
    
    参数:
        N: 接收天线数
        M: 发送天线数
        constellation: 星座点数组
        sparsity: 稀疏度（激活的天线比例）
    
    返回:
        x: 发送信号向量
        H: 信道矩阵
        y: 接收信号
    """
    # 生成信道矩阵（Rayleigh衰落）
    H = (np.random.randn(N, M) + 1j * np.random.randn(N, M)) / np.sqrt(2)
    
    # 生成稀疏信号
    x = np.zeros(M, dtype=complex)
    active_indices = np.random.choice(M, size=int(M * sparsity), replace=False)
    for idx in active_indices:
        x[idx] = np.random.choice(constellation)
    
    # 添加噪声
    noise_var = 0.01  # 噪声方差
    n = np.sqrt(noise_var / 2) * (np.random.randn(N) + 1j * np.random.randn(N))
    y = H @ x + n
    
    return x, H, y, noise_var


def compute_ser(y, H, x_true, detector_func, constellation, **kwargs):
    """
    计算符号错误率（SER）
    
    参数:
        detector_func: 检测函数
    
    返回:
        ser: 符号错误率
    """
    x_hat = detector_func(y, H, **kwargs)
    
    # 判断每个符号
    errors = 0
    for i in range(len(x_true)):
        # 找到最近的星座点
        dist_true = np.abs(constellation - x_true[i])
        dist_est = np.abs(constellation - x_hat[i])
        if np.argmin(dist_true) != np.argmin(dist_est):
            errors += 1
    
    return errors / len(x_true)


# 测试单个样本
x_true, H, y, noise_var = generate_mimo_signal(N, M, constellation_qpsk)
print(f"发送信号: {x_true}")
print(f"信道矩阵形状: {H.shape}")
print(f"接收信号: {y}")

In [ ]:
# 蒙特卡洛仿真 - 不同SNR下的性能
SNR_range = [5, 10, 15, 20, 25]  # dB
num_trials = 500

results = {
    'ZF': [],
    'MMSE': [],
    'AMP': [],
    'GAMP': []
}

for snr_db in SNR_range:
    # 设置噪声方差（对应SNR）
    snr_linear = 10**(snr_db / 10)
    noise_var = 1.0 / snr_linear
    
    ser_zf = []
    ser_mmse = []
    ser_amp = []
    ser_gamp = []
    
    for trial in range(num_trials):
        # 生成信号
        x_true, H, y, _ = generate_mimo_signal(N, M, constellation_qpsk)
        y = H @ x_true + np.sqrt(noise_var / 2) * (np.random.randn(N) + 1j * np.random.randn(N))
        
        # ZF检测
        try:
            x_zf = zf_detector_quantized(y, H, constellation_qpsk)
            ser_zf.append(np.mean(np.abs(x_zf - x_true) > 0.1))
        except:
            ser_zf.append(1.0)
        
        # MMSE检测
        try:
            x_mmse = mmse_detector_quantized(y, H, constellation_qpsk, noise_var)
            ser_mmse.append(np.mean(np.abs(x_mmse - x_true) > 0.1))
        except:
            ser_mmse.append(1.0)
        
        # AMP检测
        try:
            x_amp, _ = amp_detector(y, H, noise_var)
            x_amp_q = np.zeros_like(x_amp)
            for i in range(M):
                idx = np.argmin(np.abs(constellation_qpsk - x_amp[i]))
                x_amp_q[i] = constellation_qpsk[idx]
            ser_amp.append(np.mean(np.abs(x_amp_q - x_true) > 0.1))
        except:
            ser_amp.append(1.0)
        
        # GAMP检测
        try:
            x_gamp, _ = gamp_detector(y, H, noise_var)
            x_gamp_q = np.zeros_like(x_gamp)
            for i in range(M):
                idx = np.argmin(np.abs(constellation_qpsk - x_gamp[i]))
                x_gamp_q[i] = constellation_qpsk[idx]
            ser_gamp.append(np.mean(np.abs(x_gamp_q - x_true) > 0.1))
        except:
            ser_gamp.append(1.0)
    
    results['ZF'].append(np.mean(ser_zf))
    results['MMSE'].append(np.mean(ser_mmse))
    results['AMP'].append(np.mean(ser_amp))
    results['GAMP'].append(np.mean(ser_gamp))
    
    print(f"SNR={snr_db}dB: ZF SER={results['ZF'][-1]:.4f}, MMSE SER={results['MMSE'][-1]:.4f}, AMP SER={results['AMP'][-1]:.4f}, GAMP SER={results['GAMP'][-1]:.4f}")

In [ ]:
# 绘制BER曲线
plt.figure(figsize=(10, 6))
plt.semilogy(SNR_range, results['ZF'], 'o-', label='ZF (迫零)', linewidth=2)
plt.semilogy(SNR_range, results['MMSE'], 's-', label='MMSE (最小均方误差)', linewidth=2)
plt.semilogy(SNR_range, results['AMP'], '^-', label='AMP (近似消息传递)', linewidth=2)
plt.semilogy(SNR_range, results['GAMP'], 'd-', label='GAMP (高斯AMP)', linewidth=2)

plt.xlabel('SNR (dB)', fontsize=12)
plt.ylabel('符号错误率 (SER)', fontsize=12)
plt.title('MIMO检测器性能对比', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.ylim([1e-3, 1])
plt.tight_layout()
plt.show()

## 8. 收敛性分析 - AMP与GAMP迭代过程

In [ ]:
# 分析迭代收敛性
x_true, H, y, _ = generate_mimo_signal(N, M, constellation_qpsk)
noise_var = 0.01
y = H @ x_true + np.sqrt(noise_var / 2) * (np.random.randn(N) + 1j * np.random.randn(N))

# AMP迭代
x_amp, residuals_amp = amp_detector(y, H, noise_var, max_iter=30)

# GAMP迭代
x_gamp, snr_gamp = gamp_detector(y, H, noise_var, max_iter=30)

print(f"真实信号: {x_true}")
print(f"AMP估计: {x_amp}")
print(f"GAMP估计: {x_gamp}")

In [ ]:
# 绘制收敛曲线
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(residuals_amp, 'o-', label='AMP残差')
plt.xlabel('迭代次数', fontsize=11)
plt.ylabel('归一化残差', fontsize=11)
plt.title('AMP算法收敛性', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')

plt.subplot(1, 2, 2)
plt.plot(snr_gamp, 's-', label='GAMP估计SNR', color='orange')
plt.xlabel('迭代次数', fontsize=11)
plt.ylabel('估计SNR (dB)', fontsize=11)
plt.title('GAMP算法收敛性', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 算法复杂度对比

| 算法 | 计算复杂度 | 特点 |
|------|-----------|------|
| ML | $O(|\mathcal{X}|^M)$ | 最优但不可扩展 |
| ZF | $O(M^3)$ | 线性复杂度，实现简单 |
| MMSE | $O(M^3)$ | 性能优于ZF |
| BP | $O(k M N)$ | 迭代复杂度，适合稀疏系统 |
| AMP | $O(M N)$ per iter | 低复杂度，高性能 |
| GAMP | $O(M N)$ per iter | AMP推广，更通用 |

## 10. 思考题

1. **ZF vs MMSE**: 在低SNR条件下，ZF和MMSE哪种检测器性能更好？为什么？

2. **AMP收敛性**: AMP算法的收敛性取决于什么因素？在什么情况下AMP可能不收敛？

3. **稀疏先验**: 如果发送信号不是稀疏的，消息传递算法还能提升性能吗？为什么？

4. **大规模MIMO**: 当天线数量增加到100以上时，哪种检测算法最为实用？为什么？

5. **混合检测**: 能否设计一种结合ZF和AMP优点的混合检测器？简述其基本思路。

## 11. 深入阅读

- "Turbo Equalization" - 使用迭代检测和译码
- "Approximate Message Passing" - Donoho, Maleki, Montanari
- "Sparse Signal Recovery" - Candes, Tao, Romberg
- "Massive MIMO Detection" - Larsson, Edfors, Tufvesson